# Gradient descent for generic functions

We'll start by reasoning a little bit about how gradient descent works in various cases. We will then compute the gradient for several functions on the whiteboard. And we will wrap up by writing vectorized Numpy code for computing the gradient of a few of these functions.


## 1) The behavior of gradient descent


Let's consider a linear regression problem. For simplicty, we'll ignore the offset (assume $\theta_0=0$). Our hypothesis has the form $h(x;\theta)=\theta^\top x$. Our objective function has the form: $$J(\theta)=\frac{1}{n}\sum_{i=1}^{n}(h(x^{(i)};\theta) - y^{(i)})^2 + \lambda \|\theta\|^2\enspace.$$

Recall that one iteration of gradient descent is defined as: $$\theta^{(t)} = \theta^{(t-1)} -\eta \nabla_\theta J(\theta)\Big|_{\theta=\theta^{(t-1)}}\enspace,$$ where $\eta$ is the step size.

### 1.1) Visualizing gradient descent
Consider this 1D dataset with 4 data points:
```
X = np.array([[1],[2],[3],[4]])
y = np.array([[1.2], [1.8], [3.2], [3.8]])
```
Here are plots of the objective function for $\lambda=0$ with various trajectories of gradient descent overlaid. The trajectories may vary by the choice of the step size and starting guess.

<img src="https://drive.google.com/uc?export=view&id=1gLq6VKX_U2dDoDnTG1oZyxQx_doJ_fNr" width="300"/>
<img src="https://drive.google.com/uc?export=view&id=1q5fq0-WygqC8AdZ_jccj4ROBpY6jZYI1" width="300"/>
<img src="https://drive.google.com/uc?export=view&id=1yapZJj3bIQewe4qDIubr9X8Iyj3AkM9i" width="300"/>
<img src="https://drive.google.com/uc?export=view&id=159F4LvVEk47ZuAi9AA-Nr5qIZCVs-sY8" width="300"/>
<img src="https://drive.google.com/uc?export=view&id=1UEYplpcZDWpvy9DAJKz2cr70WOVQqjd_" width="300"/>

#### 1.1.1) **Question:** Should we expect to be able to find the global min using gradient descent? Why/why not?

#### 1.1.2) **Question:** What are the starting guesses for theta?

#### 1.1.3) **Question:** Sort the values of $\eta$ in ascending order

#### 1.1.4) **Question:** For each plot: does it converge with no oscillations, converge with oscillations, has too few iterations to converge, or diverge?

### 1.2) Comparing numerical results

Suppose we have the following function calls for learning a linear regressor:

```
def lin_reg_analytic(X, y, lambda):
  ... # Implemented with the closed-form solution to theta
  return (theta, MSE)

def regress_gd(X, y, lambda, step_size, num_steps, init):
  ... # Implemented as studied today, stopping after convergence or reaching num_steps
  return (theta, MSE)
```

#### 1.2.1) Consider the following 2D dataset:
```
X = np.array([[1, 2], [2, 1], [3, 4], [4, 3]])
y = np.array([[1.2], [1.8], [3.2], [3.8]])
```

We first compute the analytic solution without regularization ($\lambda=0$) and then run three instances of gradient descent with different values for the step size:

```
>>> lin_reg_analytic(X, y, 0)
(array([0.8, 0.2], 0)


>>> regress_gd(X, y, 0.00, 0.1, 1000, np.array([ 0.0, 0.0]))
(array([-2.83511681e+278,-2.83511681e+278]), inf)

>>> regress_gd(X, y, 0.00, 0.01, 1000, np.array([ 0.0, 0.0]))
 (array([0.79998705, 0.20001295]), 1.67738094269143e-10)

>>> regress_gd(X, y, 0.00, 0.001, 1000, np.array([ 0.0, 0.0]))
(array([0.68969137, 0.31030863]), 0.012167993285775044)
```

**Question:** Compare the outputs of the three executions of GD with different hyperparameters: which converged/diverged/failed to converge?


#### 1.2.2) Now consider these two _slightly_ different datasets:
```
X1 = np.array([[1, 1], [2, 2], [3, 3], [4, 4]])
X2 = np.array([[1, 1.000001], [2, 2.000002], [3, 3.000003], [4, 4.000004]])
y1 = np.array([[1], [2], [3], [4]])
y2 = np.array([1.2], [1.8], [3.2], [3.8]])
```

##### i) **Question:** what would happen if we attempt to run `lin_reg_analytic(X1, y1, 0)`?

##### ii) **Question:** let's compare the results of running `lin_reg_analytic(X2, y1, 0)` and `lin_reg_analytic(X2, y2, 0)`:
```
>>> lin_reg_analytic(X2, y1, 0)
(array([ 9.99961689e-01,-3.63133399e-05]), 1.6706405582993305e-07)

>>> lin_reg_analytic(X2,y2,0)
(array([ 52385.11668617,-52384.03587999]), 0.051556090396712245)
```
What behavior are we observing when attempting to compute the analytic solution?


##### iii) **Question:** Let's take the second hypothesis (obtained from ` lin_reg_analytic(X2,y2,0)`). What is the predicted label for `x=[1,1]`? And for `[1.01, 1.02]`? Is that good?


#### 1.2.3) Now let's consider the effect of regularization on the same datasets of (1.2.2). Let's again compare the following runs, varying $\lambda$ cross runs of gradient descent:

```
>>> lin_reg_analytic(X2, y2, 0.01)
array([0.49299887, 0.49301017])


>>> regress_gd(X2, y2, 0.01, 0.01, 100000, np.array([ 10,-10]))
(array([0.49299889, 0.49301015]), 0.04353086796908341)

>>> regress_gd(X2, y2, 0.0, 0.01, 100000, np.array([ 10,-10]))
(array([10.49322603,-9.50655365]), 0.03866875820336399)
```

**Question:** What impact does regularization have on learning a hypothesis for this dataset?


## 1) Whiteboard computation of gradients

Let's think about doing linear regression given data $X$, $y$ to find parameters $\theta$, $\theta_0$ that minimize the MSE: $$J(\theta) = \frac{1}{n} \sum_{i=1}^n (\theta^\top x^{(i)} - y^{(i)})^2\enspace.$$

The data is shaped like: $$X = \begin{bmatrix} x_1^{(1)} & \cdots & x_d^{(1)} \\ \vdots & \ddots & \vdots \\ x_1^{(n)} & \cdots & x_d^{(n)} \end{bmatrix}, \quad Y = \begin{bmatrix} y^{(1)} \\ \vdots \\ y^{(n)} \end{bmatrix}\enspace.$$

(_Note: we are assuming that $\theta_0$ is part of $\theta$, and that $X$ has a column of all-ones. Also note that while $x^{(i)}$ makes up the $i$-th row of $X$, here we treat it as a column vector._)

### 2.1) **Question:** If $X$ is $n\times d$ and $Y$ is $n \times 1$, what is the dimension of $\theta$?


### 2.2) **Question**: What is the dimension of $\nabla_\theta J(\theta, \theta_0)$?


### 2.3) **Question:** Say $L(x^{(i)}, y^{(i)}; \theta) = (\theta^\top x^{(i)} - y^{(i)})^2$. What is the gradient of $L$ w.r.t $\theta$?


### 2.4) **Question:** What is the gradient of $J$ w.r.t $\theta$?


### 2.5) **Question:** Write an expression for $\nabla_\theta J$ in terms of the matrices $X$ and $Y$.


### 2.6) **Question:** We can now find the analytic solution for $\theta^*$. Let's work it out:

## 3) Numpy computation of gradients

Let's now write code for the expressions we found above

In [ ]:
import numpy as np

In [ ]:
X = np.random.randn(100, 30)  # what is n? d?
Y = np.random.randn(100, 1)
theta = np.random.randn(30, 1)

In [ ]:
# pick an arbitrary i, say i=21
i = 21
x_i = ...   # get the i-th element from X and store as a column vector. From HW1, recall that we can use slicing to get 1 single element and preserve dimensions.
y_i = ...        # get the i-th element from Y and store as a 1x1 vector
grad_Li = ...   # compute the gradient of L(x_i, y_i; theta) w.r.t theta

In [ ]:
grad_J = ...   # compute thegradient of J(X, Y; theta) w.r.t theta

In [ ]:
theta_star = ...   # find theta*